In [10]:
import torch
from transformers import AutoTokenizer, EsmModel
from sklearn.metrics.pairwise import cosine_similarity
import torch.nn.functional as F

In [3]:
p53 = "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"

In [6]:
# which amino acid that appear many times
print("\n" + "="*30)
unique_aas = set(p53)
print("Counting amino acids:")
for aa in unique_aas:
    count = p53.count(aa)
    print(f"{aa}: {count}")


Counting amino acids:
F: 11
A: 24
K: 20
L: 32
P: 45
Y: 9
E: 30
Q: 15
T: 22
I: 8
S: 38
C: 10
W: 4
D: 20
R: 26
G: 23
H: 12
V: 18
N: 14
M: 12


In [8]:
# Find all P positions (1-indexed)
p_positions = [i+1 for i, aa in enumerate(p53) if aa == 'P']
print(f"Total P residues: {len(p_positions)}")
print(f"P positions: {p_positions[:]}...")  # First 20

Total P residues: 45
P positions: [4, 8, 12, 13, 27, 34, 36, 47, 58, 60, 64, 67, 71, 72, 75, 77, 80, 82, 85, 87, 89, 92, 98, 128, 142, 151, 152, 153, 177, 190, 191, 219, 222, 223, 250, 278, 295, 300, 301, 309, 316, 318, 322, 359, 390]...


In [11]:
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name)
model.eval()

inputs = tokenizer(p53, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
emb = outputs.last_hidden_state[0]
p1 = emb[4]
p2 = emb[250]

similarity = F.cosine_similarity(p1, p2, dim=0)
print(similarity.item())

0.55326908826828


In [13]:
p3 = emb[4]
p4 = emb[8]
similarity = F.cosine_similarity(p3, p4, dim=0)
print(similarity.item())

0.9570600986480713


## comparing full tp53 and certain region removes

In [14]:
full_emb = outputs.last_hidden_state
full_mask = inputs["attention_mask"]

full_protein = (
    full_emb * full_mask.unsqueeze(-1)
).sum(dim=1) / full_mask.sum(dim=1, keepdim=True)

# remove residue
mask2 = full_mask.clone()
mask2[:, 240:260] = 0 # remove region

protein_removed = (
    full_emb * mask2.unsqueeze(-1)
).sum(dim=1) / mask2.sum(dim=1, keepdim=True)

similarity = F.cosine_similarity(full_protein, protein_removed)
print(similarity.item())

0.9997043013572693


## Residue Importance
###  Residue influence experiment
**What we will do**

For a small number of residues (not all yet):
1. Compute the full protein embedding
2. Remove one residue
3. Recompute the embedding
4. Measure the change

The magnitude of change = importance score

In [18]:
emb = outputs.last_hidden_state
mask = inputs["attention_mask"]

# full protein embedding
full_protein = (
    emb * mask.unsqueeze(-1)
).sum(dim=1) / mask.sum(dim=1, keepdim=True)


def protein_without_residue(idx):
    mask2 = mask.clone()
    mask2[:, idx] = 0
    return (
        emb * mask2.unsqueeze(-1)
    ).sum(dim=1) / mask2.sum(dim=1, keepdim=True)

idx = 250
protein_minus = protein_without_residue(idx)

importance = 1 - F.cosine_similarity(full_protein, protein_minus)
print(importance.item())

2.86102294921875e-06


In [20]:
idx2 = 4
protein_minus2 = protein_without_residue(idx2)
importance2 = 1 - F.cosine_similarity(full_protein, protein_minus2)
print(importance2.item())

7.748603820800781e-07


## Fixing Aggregation — Step 4 (First Constructive Solution)

In [31]:
emb = outputs.last_hidden_state[0]
mask = inputs["attention_mask"][0].bool()

# remove special token 
emb_valid = emb[mask]

# compute importance weights from embedding norms
weights = torch.norm(emb_valid, dim=1)
weights = weights / weights.sum()

# weighted_pooling
full_protein_weighted = (emb_valid * weights.unsqueeze(1)).sum(dim=0)

print(full_protein_weighted.shape)

torch.Size([1280])


In [32]:
# make sure everything is consistent
print("Mean pooling shape:",
      outputs.last_hidden_state.mean(dim=1).shape)

print("Weighted pooling shape:",
      full_protein_weighted.unsqueeze(0).shape)


Mean pooling shape: torch.Size([1, 1280])
Weighted pooling shape: torch.Size([1, 1280])


In [37]:
# Protein embedding with one residue removed (weighted)
def protein_without_residue_weighted(idx):
    mask2 = mask.clone()
    mask2[idx] = 0

    emb2 = emb[mask2]
    weights2 = torch.norm(emb2, dim=1)
    weights2 = weights2 / weights2.sum()

    return (emb2 * weights2.unsqueeze(1)).sum(dim=0)



In [38]:
# Importance measurement
idx = 250

protein_wighted_minus = protein_without_residue_weighted(idx)

importance = 1 - F.cosine_similarity(full_protein_weighted.unsqueeze(0), protein_wighted_minus.unsqueeze(0))

print(importance.item())

2.4437904357910156e-06


## Seeing importance along the TP53 sequence

In [40]:
positions = list(range(5, emb.shape[0] - 5, 5))

This means:
- every 5 residues
- skip edges / special tokens
- fast and interpretable



#### Compute importance per position

In [41]:
importance_scores = []

for idx in positions:
    protein_minus = protein_without_residue_weighted(idx)
    imp = 1 - F.cosine_similarity(
        full_protein_weighted.unsqueeze(0),
        protein_minus.unsqueeze(0)
    )
    importance_scores.append(imp.item())

In [ ]:
# plot
import matplotlib.pyplot as plt